In [ ]:
import lightgbm as lgb
print(lgb.__version__)
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.datasets import load_iris

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = LGBMClassifier(
    n_estimators=100,
    learning_rate=0.1,
    num_leaves=31,
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_test)
proba = model.predict_proba(X_test)

print("acc:", accuracy_score(y_test, pred))
print("feature importance:", model.feature_importances_)

In [ ]:
from lightgbm import LGBMRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.datasets import fetch_california_housing

X, y = fetch_california_housing(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LGBMRegressor(
    n_estimators=200,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_test)
rmse = mean_squared_error(y_test, pred) ** 0.5
print("rmse:", rmse)

max_depth

트리 최대 깊이.

max_depth=5
트리 깊이 제한
-1이면 제한 없음
과적합 방지용
min_child_samples

리프 노드에 있어야 하는 최소 샘플 수.

min_child_samples=20
클수록 덜 복잡해짐
과적합 방지에 도움
subsample

행 샘플 비율. bagging 비슷하다.

subsample=0.8
80% 샘플만 써서 트리 생성
과적합 감소
colsample_bytree

컬럼 샘플 비율.

colsample_bytree=0.8
트리 만들 때 feature 일부만 사용
과적합 감소
reg_alpha

L1 규제.

reg_alpha=0.1
reg_lambda

L2 규제.

reg_lambda=0.1

공식 문서도 L1/L2 regularization을 지원한다고 명시한다.

objective

문제 종류 지정.

objective="binary"
objective="multiclass"
objective="regression"
이진분류
다중분류
회귀
random_state

랜덤 시드.

random_state=42
n_jobs

병렬 스레드 수.

n_jobs=-1

LGBMClassifier 문서에 n_jobs는 병렬 thread 수라고 되어 있다.


7. feature importance

importance_type은 보통 split 또는 gain을 쓴다.
공식 문서 기준으로:

split: 해당 변수가 몇 번 split에 쓰였는지
gain: 해당 변수가 만든 split의 총 이득
model = LGBMClassifier(importance_type="gain")
model.fit(X_train, y_train)

print(model.feature_importances_)

해석은 보통 이렇게 한다.

값이 클수록 중요한 변수
split은 사용 횟수 중심
gain은 성능 기여 중심

실무에선 gain이 더 해석하기 편한 경우가 많다.

8. 결측치 처리

LightGBM은 기본적으로 missing value를 처리할 수 있고, NaN을 결측으로 다룬다. use_missing=false로 끌 수 있고, zero_as_missing=true로 0을 결측처럼 볼 수도 있다.

즉, 시험에서는 보통 이렇게 적으면 된다.

# LightGBM은 NaN 결측치를 기본적으로 처리 가능
model.fit(X_train, y_train)


장점: 빠름, 대용량 처리 좋음, 결측치 처리 가능, 성능 좋음
가장 큰 차이는 트리 성장 방식이다. LightGBM은 leaf-wise 성장을 쓰고, 이 방식은 보통 더 빨리 수렴할 수 있지만 데이터가 작거나 튜닝이 거칠면 과적합이 더 쉽게 날 수 있다고 공식 문서도 설명한다. 반면 XGBoost는 tree_method를 통해 exact, approx, hist 같은 학습 방식을 제공한다.3. 범주형 처리에 제약이 있음
LightGBM은 categorical feature를 지원하지만, 음수 값은 결측으로 처리되고, categorical feature에 대해 monotonic constraint를 줄 수 없다고 문서에 나와 있다. 또 큰 정수 범주값은 메모리를 더 먹을 수 있어서 보통 0부터 시작하는 연속 정수 인코딩을 권장한다.

| 항목      | LightGBM     | XGBoost         |
| ------- | ------------ | --------------- |
| 학습 속도   | 대체로 빠름       | 대체로 더 느림        |
| 과적합 위험  | 더 큼          | 상대적으로 안정적       |
| 대용량 데이터 | 강함           | 좋지만 속도는 밀릴 수 있음 |
| 작은 데이터  | 튜닝 민감        | 상대적으로 무난        |
| 설치 편의   | OpenMP 이슈 가능 | 보통 더 무난한 편      |
| 범주형 처리  | 지원, 제약 있음    | 지원              |
| 결측치 처리  | 지원           | 지원              |
